# Step 5 — Handle Class Imbalance

Our credit-risk dataset has a **~22 % default rate** (1 = default, 0 = no default).  
A naïve classifier can reach ~78 % accuracy by always predicting 0 — clearly useless.

| # | Section | Goal |
|---|---------|------|
| 1 | **Quantify the imbalance** | Visualise and measure the class distribution |
| 2 | **Baseline model (no balancing)** | Logistic Regression with default settings |
| 3 | **Balanced model (`class_weight='balanced'`)** | Same model, one parameter change |
| 4 | **Side-by-side comparison** | Precision / Recall / F1 / ROC-AUC |
| 5 | **Why not SMOTE (yet)?** | Conceptual overview & leakage risk |

> **Key takeaway:** `class_weight='balanced'` is safe, simple, and effective.  
> SMOTE is powerful but must be applied *inside* each CV fold — we'll revisit it later.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 12, 'axes.labelsize': 10})

RANDOM_STATE = 42
print('Libraries loaded ✓')

---
## 1 · Quantify the Imbalance

In [ ]:
DATA_PATH = pathlib.Path('..') / 'data' / 'raw' / 'credit_risk_dataset.csv'
TARGET = 'loan_status'

df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
print(f'\nTarget distribution:')
counts = df[TARGET].value_counts()
pcts   = df[TARGET].value_counts(normalize=True) * 100
summary = pd.DataFrame({'count': counts, 'pct': pcts.round(2)})
summary.index.name = TARGET
print(summary)
print(f'\nImbalance ratio (majority / minority): {counts[0] / counts[1]:.2f} : 1')

In [ ]:
# ── Visual: class distribution ──
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar chart
colours = ['#3498db', '#e74c3c']
axes[0].bar(summary.index.astype(str), summary['count'], color=colours, edgecolor='white')
for i, (cnt, pct) in enumerate(zip(summary['count'], summary['pct'])):
    axes[0].text(i, cnt + 300, f'{cnt:,}\n({pct:.1f}%)', ha='center', fontsize=10, fontweight='bold')
axes[0].set_xlabel('loan_status')
axes[0].set_ylabel('Count')
axes[0].set_title('Class Distribution — Bar Chart')
axes[0].set_xticklabels(['0 (No Default)', '1 (Default)'])

# Pie chart
axes[1].pie(summary['count'], labels=['No Default (0)', 'Default (1)'],
            autopct='%1.1f%%', colors=colours, startangle=90,
            explode=(0, 0.06), textprops={'fontsize': 11})
axes[1].set_title('Class Distribution — Pie Chart')

fig.suptitle('Target Variable Imbalance', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('08_class_imbalance.png', bbox_inches='tight')
plt.show()
print('Saved → 08_class_imbalance.png')

### What does this imbalance mean in practice?

With ~78 % of loans being non-default:
- A **dummy classifier** that always predicts 0 gets **78 % accuracy** — misleading!
- The model will be biased toward the majority class unless we intervene.
- **Recall for defaults** (class 1) will suffer — the very thing we care about.

**Solution:** Tell the model that misclassifying a default is *more costly* than
misclassifying a non-default → `class_weight='balanced'`.

---
## 2 · Prepare Data for Modelling

In [ ]:
# ── Preprocessing ────────────────────────────────────────────────────
df_model = df.copy()

# Impute missing values
for col in df_model.columns:
    if df_model[col].dtype in ['float64', 'int64']:
        df_model[col] = df_model[col].fillna(df_model[col].median())
    else:
        df_model[col] = df_model[col].fillna(df_model[col].mode()[0])

# Label-encode categoricals
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
le_dict  = {}
for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    le_dict[col] = le

# Split features / target
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

# Train / test split  (stratified to preserve class ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train default rate: {y_train.mean()*100:.2f}%')
print(f'Test  default rate: {y_test.mean()*100:.2f}%')

---
## 3 · Baseline vs Balanced — Logistic Regression

In [ ]:
# ── Train two models ─────────────────────────────────────────────────
lr_baseline = LogisticRegression(
    max_iter=1000, random_state=RANDOM_STATE
)
lr_balanced = LogisticRegression(
    max_iter=1000, random_state=RANDOM_STATE,
    class_weight='balanced'          # ← the ONE change
)

lr_baseline.fit(X_train_sc, y_train)
lr_balanced.fit(X_train_sc, y_train)
print('Both models trained ✓')

### How `class_weight='balanced'` works

Sklearn computes per-class weights automatically:

$$w_c = \frac{n_{\text{samples}}}{n_{\text{classes}} \times n_{\text{samples in class } c}}$$

This makes the minority class (defaults) contribute more to the loss,
forcing the model to pay attention to them.

In [ ]:
# ── Show the computed weights ────────────────────────────────────────
n_samples = len(y_train)
n_classes = 2
for cls in [0, 1]:
    n_cls = (y_train == cls).sum()
    w = n_samples / (n_classes * n_cls)
    print(f'  class {cls}  →  weight = {w:.4f}   (n={n_cls:,})')
print('\nMinority class (1) gets a higher weight → penalises misses more.')

---
## 4 · Side-by-Side Comparison

In [ ]:
# ── Classification reports ───────────────────────────────────────────
y_pred_base = lr_baseline.predict(X_test_sc)
y_pred_bal  = lr_balanced.predict(X_test_sc)

print('=' * 60)
print('BASELINE  (no class_weight)')
print('=' * 60)
print(classification_report(y_test, y_pred_base, target_names=['No Default', 'Default']))

print('=' * 60)
print('BALANCED  (class_weight="balanced")')
print('=' * 60)
print(classification_report(y_test, y_pred_bal, target_names=['No Default', 'Default']))

In [ ]:
# ── Confusion matrices side by side ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, preds, title in [
    (axes[0], y_pred_base, 'Baseline (no balancing)'),
    (axes[1], y_pred_bal,  'Balanced (class_weight)'),
]:
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Default', 'Default'])
    disp.plot(ax=ax, cmap='Blues', values_format=',')
    ax.set_title(title, fontsize=12, fontweight='bold')

fig.suptitle('Confusion Matrix Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('09_confusion_matrices.png', bbox_inches='tight')
plt.show()
print('Saved → 09_confusion_matrices.png')

In [ ]:
# ── ROC & Precision-Recall curves ────────────────────────────────────
y_prob_base = lr_baseline.predict_proba(X_test_sc)[:, 1]
y_prob_bal  = lr_balanced.predict_proba(X_test_sc)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
for probs, label, colour in [
    (y_prob_base, 'Baseline', '#3498db'),
    (y_prob_bal,  'Balanced', '#e74c3c'),
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    axes[0].plot(fpr, tpr, label=f'{label} (AUC={auc:.3f})', color=colour, lw=2)
axes[0].plot([0,1], [0,1], 'k--', lw=0.8, alpha=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend(loc='lower right')

# Precision-Recall
for probs, label, colour in [
    (y_prob_base, 'Baseline', '#3498db'),
    (y_prob_bal,  'Balanced', '#e74c3c'),
]:
    prec, rec, _ = precision_recall_curve(y_test, probs)
    ap = average_precision_score(y_test, probs)
    axes[1].plot(rec, prec, label=f'{label} (AP={ap:.3f})', color=colour, lw=2)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='upper right')

fig.suptitle('Baseline vs Balanced — Curve Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('10_roc_pr_curves.png', bbox_inches='tight')
plt.show()
print('Saved → 10_roc_pr_curves.png')

In [ ]:
# ── Summary metrics table ────────────────────────────────────────────
from sklearn.metrics import precision_score, recall_score, f1_score

rows = []
for name, preds, probs in [
    ('Baseline',  y_pred_base, y_prob_base),
    ('Balanced',  y_pred_bal,  y_prob_bal),
]:
    rows.append({
        'Model': name,
        'Precision (Default)': precision_score(y_test, preds),
        'Recall (Default)':    recall_score(y_test, preds),
        'F1 (Default)':        f1_score(y_test, preds),
        'ROC-AUC':             roc_auc_score(y_test, probs),
        'Avg Precision':       average_precision_score(y_test, probs),
    })

metrics_df = pd.DataFrame(rows).set_index('Model').round(4)
print('\n📊 Metrics comparison (class 1 = Default):\n')
metrics_df

### Key observations

| Metric | Baseline | Balanced | Winner |
|--------|----------|----------|--------|
| **Recall (Default)** | Low | Higher | ✅ Balanced |
| **Precision (Default)** | Higher | Lower | ⚠️ Trade-off |
| **F1 (Default)** | — | — | Usually Balanced |
| **ROC-AUC** | Similar | Similar | ≈ Tie |

The balanced model **catches more actual defaults** (higher recall) at the cost of
slightly more false alarms. In credit risk, **missing a default is far costlier** than
a false alarm, so this trade-off is usually worthwhile.

---
## 5 · Cross-Validation Sanity Check

Verify the balanced model generalises — not just lucky on one split.

In [ ]:
# ── 5-fold stratified CV ─────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Scale full X for CV
X_all_sc = StandardScaler().fit_transform(X)

cv_results = {}
for name, model in [
    ('Baseline',  LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ('Balanced',  LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced')),
]:
    scores = cross_validate(
        model, X_all_sc, y, cv=cv,
        scoring=['f1', 'recall', 'roc_auc', 'precision'],
        return_train_score=False,
    )
    cv_results[name] = scores
    print(f'\n{name}:')
    for metric in ['f1', 'recall', 'roc_auc', 'precision']:
        vals = scores[f'test_{metric}']
        print(f'  {metric:12s}  →  {vals.mean():.4f} ± {vals.std():.4f}')

print('\n✓ Cross-validation complete')

In [ ]:
# ── CV box-plot comparison ───────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)

for i, metric in enumerate(['f1', 'recall', 'roc_auc', 'precision']):
    data = [cv_results['Baseline'][f'test_{metric}'],
            cv_results['Balanced'][f'test_{metric}']]
    bp = axes[i].boxplot(data, labels=['Baseline', 'Balanced'],
                         patch_artist=True, widths=0.5)
    bp['boxes'][0].set_facecolor('#3498db')
    bp['boxes'][1].set_facecolor('#e74c3c')
    axes[i].set_title(metric.replace('_', ' ').title(), fontweight='bold')
    axes[i].set_ylabel('Score')

fig.suptitle('5-Fold Stratified CV — Baseline vs Balanced', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('11_cv_comparison.png', bbox_inches='tight')
plt.show()
print('Saved → 11_cv_comparison.png')

---
## 6 · Why Not SMOTE (Yet)?

**SMOTE** (Synthetic Minority Oversampling Technique) generates synthetic samples
for the minority class by interpolating between existing data points.

### How SMOTE works (conceptually)
1. Pick a minority-class sample
2. Find its *k* nearest minority-class neighbours
3. Create a new synthetic point on the line segment between them
4. Repeat until the classes are balanced

### Why we're NOT using it now

| Risk | Explanation |
|------|-------------|
| **Data leakage** | If you SMOTE *before* train/test split, synthetic points based on test data leak into training — inflating metrics |
| **Must go inside CV** | SMOTE must be applied inside each fold of cross-validation (via `imblearn.Pipeline`), adding complexity |
| **Noise amplification** | SMOTE can create synthetic samples near the decision boundary from noisy outliers |
| **Not always better** | For moderate imbalance (~22%), `class_weight` often matches or beats SMOTE |

### The safe way to use SMOTE (future notebook)
```python
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote',  SMOTE(random_state=42)),   # applied INSIDE each fold
    ('model',  LogisticRegression(max_iter=1000)),
])
# Use this pipeline with cross_val_score → no leakage
```

> **Rule of thumb:** Start with `class_weight='balanced'`.  
> Only move to SMOTE if you need better recall *and* you set up imblearn pipelines correctly.

---
## Summary & Next Steps

| Finding | Detail |
|---------|--------|
| **Imbalance ratio** | ~3.6 : 1 (non-default : default), 21.8% default rate |
| **Baseline weakness** | Low recall on defaults — misses actual defaults |
| **`class_weight='balanced'`** | Boosts recall significantly with one parameter |
| **SMOTE** | Understood conceptually; deferred to avoid leakage risks |
| **CV validation** | Balanced model results are stable across 5 folds |

### Decisions made
- ✅ Use `class_weight='balanced'` as the default for all future models
- ✅ Always use stratified splits to preserve class ratios
- ⏳ SMOTE to be explored later inside `imblearn.Pipeline`

**Proceed to →** next modelling step (feature selection & model comparison)